### 1. 基于已有tools和data content生成Purposes

In [23]:
# 加载已生成的 tools.py
with open(GENERATED_CODE_DIR / "tools.py", 'r', encoding='utf-8') as f:
    tools_code = f.read()
with open(GENERATED_DATA_DIR / "db.json", 'r', encoding='utf-8') as f:
    base_content = f.read()

# 内联 prompt：基于 tools 让 LLM 提议 BASE 域的常见使用目的
purposes_prompt = f"""You are designing benchmark tasks for the BASE domain of an office collaboration system (like DingTalk/Feishu at a Chinese tech company).
Tools of the base domains:
```
{tools_code}
```

Database content:
```
{base_content}
```
Propose up to 20 realistic purposes (stop early if you run out of meaningful ones) from the perspective of actual office workers.
Think from the POV of: HR staff, team lead, IT admin, new employee, or any regular employee.
Hard constraints:
1. Each purpose must be solvable using ONLY the above BASE tools — no HR processes (leave/payroll/onboarding), no IT operations (hardware/software), no admin services (travel/rooms).
2. Each purpose must require at least 2 tool calls to resolve.
3. Prefer action-oriented tasks (more than search) over pure Q&A.
4. Do not generate duplicate (even in semantic meaning) purposes.
5. Make sure the purpose can be solved by current tools and existing data.
Output as JSON list:
[
  {{
    "purpose": "简短目的描述（中文，5字以内）",
    "queries": ["口语化表达1", "口语化表达2"],
    "persona": "谁会有这个需求（例：HR、团队负责人、普通员工）"
  }},
  ...
]
"""

print("正在提议 BASE 域 Purposes...")
# use stream mode
purposes_response = ""
for chunk in llm.llm.stream(purposes_prompt):
    purposes_response += chunk.content
    print(chunk.content, end="", flush=True)
print()
purposes_response = extract_code_block(purposes_response)
is_json, purposes_data = verify_json_structure(purposes_response)
if not is_json:
    print("✗ JSON 结构验证失败，请检查生成的 Schema")
pure_purposes = [x["purpose"] for x in purposes_data]
print(pure_purposes)


正在提议 BASE 域 Purposes...
```json
[
  {
    "purpose": "查看并标已读通知",
    "queries": ["帮我看看我有什么未读通知，然后全部标为已读", "我是EMP042，把我的未读消息都处理一下"],
    "persona": "普通员工"
  },
  {
    "purpose": "查团队成员详情",
    "queries": ["帮我看看核心平台项目组都有谁，把负责人的详细信息告诉我", "数据平台项目组的lead是谁？他是什么岗位的？"],
    "persona": "团队负责人"
  },
  {
    "purpose": "查下属的团队归属",
    "queries": ["帮我查一下王阳参加了哪些团队，分别是什么角色", "我想知道我下属邓莉都在哪些项目组里"],
    "persona": "团队负责人"
  },
  {
    "purpose": "分配角色给员工",
    "queries": ["把团队负责人角色分配给李云，范围是数据平台项目组，我是EMP005来操作", "帮我给朱博分配部门主管角色，scope是后端开发组"],
    "persona": "IT管理员"
  },
  {
    "purpose": "撤销员工角色",
    "queries": ["帮我把席婷婷的实习生角色撤销掉", "EMP070的角色分配需要撤销，帮我处理一下"],
    "persona": "HR"
  },
  {
    "purpose": "查上级汇报链",
    "queries": ["帮我查一下王平的汇报链，一直到最高层是谁", "我想知道李桂芝的直属领导是谁，领导的领导又是谁"],
    "persona": "新员工"
  },
  {
    "purpose": "查部门下属子部门",
    "queries": ["工程技术部下面有哪些子部门？每个子部门的负责人分别是谁？", "帮我看看数据智能部下面有几个组，各自什么状态"],
    "persona": "HR"
  },
  {
    "purpose": "查员工导师关系",
    "queries": ["帮我查一下周红梅的导师是谁，是什么类型的辅导",

### 生成 Tasks (tasks.json)

In [ ]:
task_prompt_template = """You are designing benchmark tasks for an Office System (类钉钉 HR 模块).

## Context

Tools available (the agent can ONLY use these):
```
{tool_code}
```

Base office system database (employees, departments, teams, positions — READ ONLY):
```
{base_content}
```

## Task

Generate ONE realistic benchmark task with the purpose: **{purpose}**

## Difficulty Levels (assign one to `difficulty`):
- **L1**: Single READ (parameters are directly known)
- **L2**: Multi-step READ (look up a ref first, then query details)
- **L3**: READ + WRITE (query first, then create/update based on result)
- **L4**: Multi-step WRITE with conditional logic (query result decides next action)
- **L5**: Adversarial/error handling (quota exceeded, permission denied, invalid params)

## Rules

**Actions (CRITICAL):**
- Every argument value containing a `_ref` or ID MUST be a real value from the provided database content above.
- Do NOT invent or hallucinate any IDs. Cross-check against `{base_content}`.
- Use `compare_args` to specify which arguments are the essential ones to check (e.g., skip optional note/comment fields).
- Each action must map to a real function in `{tool_code}`.

**initial_state:**
- For pure READ tasks (L1/L2): set to `null`.
- For any task with WRITE operations (L3/L4/L5): MUST describe the prerequisite DB state. Example: "员工 EMP022 有一条状态为 pending 的请假申请（leave_ref: 'LV-00123'）".
- The initial_state description should be specific enough that the evaluation environment can verify the starting condition.

**nl_assertions:**
- Write 2-4 short, precise, verifiable statements about what MUST be true after the task is completed.
- Use concrete values from the database (names, IDs, amounts).
- Good: "员工张伟的2024年年假余额已减少5天，当前余额为8天"
- Bad: "请假申请已提交" (too vague)

## Output Format

Use the following format to output:
```
{
    "id": "...",
    "description": {
      "purpose": "",
      "relevant_policies": "... or null",
      "notes": "... or null"
    },
    "user_scenario": {
      "persona": "... or null",
      "instructions": {
        "task_instructions": "...",
        "reason_for_call": "...",
        "known_info": "...",
        "unknown_info": "...",
        "domain": "BASE"
      }
    },
    "initial_state": "... or null",
    "evaluation_criteria": {
      "actions": [
        {
          "arguments": {
             "parameter1": value1,
             "parameter2": value2,
             ...
          },
          "name": "...",
          "action_id": "xxx"
        },
        ...
      ],
      "communicate_info": [], // key words that should appear in the communication
      "nl_assertions": []  // short sentences to describe a fact if the user request is resolved
    }
}
```
"""

# 人工筛选后填写 purposes 列表（从上一步 LLM 提议中选取）
purposes = ['查看并标已读通知', '查团队成员详情', '查下属的团队归属', '分配角色给员工', '撤销员工角色', '查上级汇报链', '查部门下属子部门', '查员工导师关系', '查岗位级别对比', '统计部门人数', '查角色权限详情', '查公司配置信息', '查员工所属部门', '查某人直接下属', '查特定岗位族员工', '标记单条通知已读', '查实习生分布', '查员工多团队情况', '给新人分配实习角色', '查离职人员信息']

all_tasks = []

for purpose in purposes:
    print(f"\n{'='*60}")
    print(f"正在生成 purpose: {purpose}")
    task_prompt = task_prompt_template
    task_prompt = task_prompt.replace("{tool_code}", tools_code)
    task_prompt = task_prompt.replace("{base_content}", base_content)
    task_prompt = task_prompt.replace("{purpose}", purpose)

    task_response = llm.generate(task_prompt)
    task_text = extract_code_block(task_response)
    print(task_text)

    is_valid, task_data = verify_json_structure(task_text)
    if is_valid:
        # LLM 可能返回单个对象或列表，统一处理为列表
        if isinstance(task_data, dict):
            task_data = [task_data]
        all_tasks.extend(task_data)
        print(f"✓ 生成成功，共 {len(task_data)} 个任务")
    else:
        print(f"✗ JSON 解析失败: {task_data}")
        print(f"  原始输出片段: {task_text[:200]}")
    print('='*60)

# 保存合并后的 tasks.json
tasks_json_text = json.dumps(all_tasks, ensure_ascii=False, indent=2)
tasks_path = GENERATED_DATA_DIR / "tasks.json"
save_generated_file(tasks_path, tasks_json_text)

print(f"\n✓ 共生成 {len(all_tasks)} 个任务，已保存到: {tasks_path}")


正在生成 purpose: 查看并标已读通知
{
    "id": "notif_read_001",
    "description": {
      "purpose": "查看并标已读通知",
      "relevant_policies": null,
      "notes": "用户想查看自己的未读通知，了解内容后将所有未读通知标记为已读"
    },
    "user_scenario": {
      "persona": "房玉珍，基础架构组的一名基础架构工程师",
      "instructions": {
        "task_instructions": "你是房玉珍，你想查看自己有哪些未读通知，了解通知内容后，把所有未读通知都标记为已读。",
        "reason_for_call": "最近工作比较忙，好几天没看系统通知了，想集中处理一下未读通知，看看有没有重要的审批请求或公告。",
        "known_info": "你的名字是房玉珍，你在基础架构组工作。",
        "unknown_info": "你不知道自己的employee_ref，也不知道具体有多少条未读通知。",
        "domain": "BASE"
      }
    },
    "initial_state": null,
    "evaluation_criteria": {
      "actions": [
        {
          "arguments": {
            "display_name": "房玉珍"
          },
          "name": "find_employee_ref_by_name",
          "action_id": "action_1"
        },
        {
          "arguments": {
            "employee_ref": "EMP024",
            "unread_only": true
          },
          "name": "list_notifications_for_employee",
